# 02. Baseline Experiment

This notebook trains a **spectrogram-as-image EfficientNet baseline**.
It supports the report sections on spectrogram-based classification, transfer learning, loss functions, and baseline experimental results.

In [ ]:
# Common setup
from pathlib import Path
import sys
import os
import json
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Find project root: works when notebook is run from repo root or from notebooks/
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    # Walk upward in case the notebook is launched from a subfolder
    for parent in Path.cwd().resolve().parents:
        if (parent / "src").exists():
            PROJECT_ROOT = parent
            break

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR = PROJECT_ROOT / "data"
REPORT_DIR = PROJECT_ROOT / "reports"
FIG_DIR = REPORT_DIR / "figures"
TABLE_DIR = REPORT_DIR / "tables"
EXP_DIR = PROJECT_ROOT / "experiments" / "notebook_outputs"
for p in [FIG_DIR, TABLE_DIR, EXP_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Source path:", SRC_DIR)

import torch
from torch.utils.data import DataLoader

from bioacoustic.dataset import read_metadata, build_class_list, add_stratified_folds, BirdAudioDataset, encode_multihot, make_label_map
from bioacoustic.models import build_model
from bioacoustic.losses import focal_bce_loss, compute_pos_weight
from bioacoustic.training import train_one_epoch, validate, save_checkpoint
from bioacoustic.utils import seed_everything, get_device

seed_everything(42)
DEVICE = get_device(True)
print("Device:", DEVICE)

## Configuration

In [ ]:
CFG = {
    "metadata_path": DATA_DIR / "train_metadata.csv",
    "audio_dir": DATA_DIR / "train_audio",
    "output_dir": EXP_DIR / "baseline",
    "sample_rate": 32000,
    "duration": 5.0,
    "n_fft": 2048,
    "hop_length": 768,
    "n_mels": 192,
    "f_min": 50,
    "f_max": 15000,
    "backbone": "tf_efficientnet_b0_ns",
    "model_type": "baseline",
    "batch_size": 16,
    "num_workers": 2,
    "epochs": 3,
    "lr": 1e-4,
    "fold": 0,
    "n_splits": 5,
    "max_train_rows": None,   # set to e.g. 1000 for quick debug
    "max_valid_rows": None,
    "label_smoothing": 0.005,
    "focal_gamma": 2.0,
}
CFG["output_dir"].mkdir(parents=True, exist_ok=True)
CFG

## Load metadata and build split

In [ ]:
df = read_metadata(CFG["metadata_path"])
classes = build_class_list(df)
label_map = make_label_map(classes)
num_classes = len(classes)
print("Num classes:", num_classes)

if "fold" not in df.columns:
    df = add_stratified_folds(df, n_splits=CFG["n_splits"], seed=42)

train_df = df[df["fold"] != CFG["fold"]].reset_index(drop=True)
valid_df = df[df["fold"] == CFG["fold"]].reset_index(drop=True)

if CFG["max_train_rows"]:
    train_df = train_df.sample(min(CFG["max_train_rows"], len(train_df)), random_state=42).reset_index(drop=True)
if CFG["max_valid_rows"]:
    valid_df = valid_df.sample(min(CFG["max_valid_rows"], len(valid_df)), random_state=42).reset_index(drop=True)

print("Train rows:", len(train_df), "Valid rows:", len(valid_df))

## Dataset and dataloader

In [ ]:
spec_kwargs = dict(
    n_fft=CFG["n_fft"],
    hop_length=CFG["hop_length"],
    n_mels=CFG["n_mels"],
    f_min=CFG["f_min"],
    f_max=CFG["f_max"],
)

train_ds = BirdAudioDataset(
    train_df,
    CFG["audio_dir"],
    classes,
    sample_rate=CFG["sample_rate"],
    duration=CFG["duration"],
    train=True,
    spectrogram_kwargs=spec_kwargs,
)
valid_ds = BirdAudioDataset(
    valid_df,
    CFG["audio_dir"],
    classes,
    sample_rate=CFG["sample_rate"],
    duration=CFG["duration"],
    train=False,
    spectrogram_kwargs=spec_kwargs,
)

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True, num_workers=CFG["num_workers"], pin_memory=True)
valid_loader = DataLoader(valid_ds, batch_size=CFG["batch_size"], shuffle=False, num_workers=CFG["num_workers"], pin_memory=True)

x, y = next(iter(train_loader))
print("Batch spec:", x.shape, "Batch target:", y.shape)

## Model, loss, and optimizer

In [ ]:
model = build_model(
    model_type=CFG["model_type"],
    num_classes=num_classes,
    backbone=CFG["backbone"],
    in_channels=1,
    pretrained=True,
).to(DEVICE)

# Compute pos_weight from training targets to reduce class imbalance.
target_matrix = []
for _, row in train_df.iterrows():
    target_matrix.append(encode_multihot(row["primary_label"], row.get("secondary_labels", None), label_map, include_secondary=True))
target_matrix = torch.tensor(np.stack(target_matrix), dtype=torch.float32)
pos_weight = compute_pos_weight(target_matrix, max_weight=20.0).to(DEVICE)

loss_fn = lambda logits, targets: focal_bce_loss(
    logits,
    targets,
    gamma=CFG["focal_gamma"],
    pos_weight=pos_weight,
    label_smoothing=CFG["label_smoothing"],
)
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=1e-4)

## Train and validate

In [ ]:
logs = []
best_auc = -1.0
for epoch in range(1, CFG["epochs"] + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device=DEVICE)
    val_out = validate(model, valid_loader, loss_fn=loss_fn, device=DEVICE)
    metrics = val_out["metrics"]
    row = {"epoch": epoch, "train_loss": train_loss, "valid_loss": val_out["loss"], **metrics}
    logs.append(row)
    print(row)

    score = metrics.get("macro_auc", float("nan"))
    if score == score and score > best_auc:
        best_auc = score
        save_checkpoint(CFG["output_dir"] / "best_baseline.pt", model, optimizer, epoch=epoch, metrics=metrics)

log_df = pd.DataFrame(logs)
display(log_df)
log_df.to_csv(CFG["output_dir"] / "training_log.csv", index=False)
np.save(CFG["output_dir"] / "valid_y_true.npy", val_out["y_true"])
np.save(CFG["output_dir"] / "valid_y_pred.npy", val_out["y_pred"])

## Plot curves

In [ ]:
if len(log_df):
    plt.figure(figsize=(8, 4))
    plt.plot(log_df["epoch"], log_df["train_loss"], marker="o", label="train loss")
    plt.plot(log_df["epoch"], log_df["valid_loss"], marker="o", label="valid loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Baseline loss curves")
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / "baseline_loss_curve.png", dpi=200)
    plt.show()

    if "macro_auc" in log_df:
        plt.figure(figsize=(8, 4))
        plt.plot(log_df["epoch"], log_df["macro_auc"], marker="o")
        plt.xlabel("Epoch")
        plt.ylabel("Macro AUC")
        plt.title("Baseline validation macro AUC")
        plt.tight_layout()
        plt.savefig(FIG_DIR / "baseline_auc_curve.png", dpi=200)
        plt.show()